In [ ]:
# -*- coding: utf-8 -*-
"""Network_pruning_on_Multiple_Datasets.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/12FXEDw9Vm0Icc2psRhGI6qCcAugXVEBP
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
from scipy.linalg import eigh
import copy

class SpectralPruner:
    """
    Network pruning using spectral graph theory to identify structurally
    important neurons via the Fiedler vector.
    """

    def __init__(self, model, device='cpu'):
        self.model = model
        self.device = device
        self.activation_storage = {}
        self.hooks = []

    def register_hooks(self, layer_names):
        """Register forward hooks to capture activations for specified layers."""
        def get_activation(name):
            def hook(module, input, output):
                self.activation_storage[name] = output.detach()
            return hook

        for name, module in self.model.named_modules():
            if name in layer_names:
                hook = module.register_forward_hook(get_activation(name))
                self.hooks.append(hook)

    def remove_hooks(self):
        """Remove all registered hooks."""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

    def compute_fiedler_importance(self, activations):
        """
        Compute importance scores for neurons using the Fiedler vector.

        Args:
            activations: Tensor of shape (batch_size, num_neurons)

        Returns:
            importance_scores: Array of importance scores for each neuron
        """
        activations = activations.cpu().numpy()

        # Compute correlation matrix as adjacency matrix
        corr_matrix = np.corrcoef(activations.T)

        # Handle NaN values (can occur if a neuron has zero variance)
        corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)

        # Ensure non-negative adjacency matrix
        adjacency = np.abs(corr_matrix)

        # Compute degree vector
        degree_vector = np.sum(adjacency, axis=1)
        degree_vector = np.maximum(degree_vector, 1e-8)  # Prevent division by zero

        # Compute normalized Laplacian: L = I - D^(-1/2) * A * D^(-1/2)
        D_inv_sqrt = np.diag(1.0 / np.sqrt(degree_vector))
        laplacian = np.eye(len(degree_vector)) - D_inv_sqrt @ adjacency @ D_inv_sqrt

        # Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = eigh(laplacian)

        # The Fiedler vector is the eigenvector corresponding to the second smallest eigenvalue
        fiedler_vector = eigenvectors[:, 1]

        # Use absolute value of Fiedler vector components as importance scores
        # Neurons with larger absolute values are more important (structural bottlenecks)
        importance_scores = np.abs(fiedler_vector)

        return importance_scores

    def collect_activations(self, dataloader, layer_name, num_batches=10):
        """
        Collect activations from a layer by running inference on data.

        Args:
            dataloader: DataLoader for the dataset
            layer_name: Name of the layer to collect activations from
            num_batches: Number of batches to use for activation collection

        Returns:
            activations: Collected activations
        """
        self.activation_storage = {}
        self.register_hooks([layer_name])

        self.model.eval()
        activations_list = []

        with torch.no_grad():
            for i, (data, _) in enumerate(dataloader):
                if i >= num_batches:
                    break
                data = data.to(self.device)
                _ = self.model(data)

                if layer_name in self.activation_storage:
                    activations_list.append(self.activation_storage[layer_name])

        self.remove_hooks()

        if not activations_list:
            raise ValueError(f"No activations collected for layer {layer_name}")

        # Concatenate all batches
        activations = torch.cat(activations_list, dim=0)

        return activations

    def prune_layer(self, layer_name, prune_ratio=0.2, dataloader=None, num_batches=10):
        """
        Prune a specific layer by removing the least important neurons.

        Args:
            layer_name: Name of the layer to prune (e.g., 'fc1')
            prune_ratio: Fraction of neurons to remove (0.0 to 1.0)
            dataloader: DataLoader for collecting activations
            num_batches: Number of batches to use for activation collection

        Returns:
            pruned_model: New model with pruned layer
            importance_scores: Computed importance scores
        """
        # Collect activations
        activations = self.collect_activations(dataloader, layer_name, num_batches)

        # Compute importance scores
        importance_scores = self.compute_fiedler_importance(activations)

        # Determine neurons to keep
        num_neurons = len(importance_scores)
        num_to_keep = int(num_neurons * (1 - prune_ratio))

        # Get indices of most important neurons
        important_indices = np.argsort(importance_scores)[-num_to_keep:]
        important_indices = np.sort(important_indices)  # Keep original order

        print(f"Pruning layer '{layer_name}': {num_neurons} -> {num_to_keep} neurons "
              f"({prune_ratio*100:.1f}% pruned)")

        # Create pruned model
        pruned_model = self._create_pruned_model(layer_name, important_indices)

        return pruned_model, importance_scores

    def _create_pruned_model(self, layer_name, keep_indices):
        """
        Create a new model with specified neurons removed from a layer.
        """
        # This is a simplified version for a basic MLP
        # For more complex architectures, you'd need to handle multiple layer types

        # Get the layer to prune
        layer_to_prune = None
        next_layer = None

        for name, module in self.model.named_modules():
            if name == layer_name:
                layer_to_prune = module
            # Find the next linear layer (to adjust input dimensions)
            elif layer_to_prune is not None and isinstance(module, nn.Linear):
                next_layer = module
                break

        if layer_to_prune is None:
            raise ValueError(f"Layer {layer_name} not found")

        # Create pruned weights and biases
        with torch.no_grad():
            # Prune output dimension of the target layer
            pruned_weight = layer_to_prune.weight[keep_indices, :]
            pruned_bias = layer_to_prune.bias[keep_indices] if layer_to_prune.bias is not None else None

            # Create new layer with reduced output dimension
            new_layer = nn.Linear(
                layer_to_prune.in_features,
                len(keep_indices),
                bias=(pruned_bias is not None)
            ).to(self.device)

            new_layer.weight.copy_(pruned_weight)
            if pruned_bias is not None:
                new_layer.bias.copy_(pruned_bias)

            # Adjust next layer's input dimension if it exists
            if next_layer is not None:
                next_layer_weight = next_layer.weight[:, keep_indices]
                new_next_layer = nn.Linear(
                    len(keep_indices),
                    next_layer.out_features,
                    bias=(next_layer.bias is not None)
                ).to(self.device)
                new_next_layer.weight.copy_(next_layer_weight)
                if next_layer.bias is not None:
                    new_next_layer.bias.copy_(next_layer.bias)

        # Build the pruned model
        # This assumes a simple sequential structure
        pruned_model = self._rebuild_model(layer_name, new_layer, next_layer, new_next_layer)

        return pruned_model

    def _rebuild_model(self, pruned_layer_name, new_layer, old_next_layer, new_next_layer):
        """Rebuild the model with the pruned layer."""
        # For simplicity, assuming a basic MLP structure
        # You may need to customize this for your specific architecture

        class PrunedMLP(nn.Module):
            def __init__(self, original_model, pruned_name, new_l, new_next_l):
                super().__init__()
                self.flatten = nn.Flatten()

                # Copy layers, replacing the pruned one
                if pruned_name == 'fc1':
                    self.fc1 = new_l
                    self.relu = nn.ReLU()
                    self.dropout = nn.Dropout(0.5)
                    self.fc2 = new_next_l if new_next_l else original_model.fc2
                else:
                    # Handle other layers if needed
                    self.fc1 = original_model.fc1
                    self.relu = nn.ReLU()
                    self.dropout = nn.Dropout(0.5)
                    self.fc2 = original_model.fc2

            def forward(self, x):
                x = self.flatten(x)
                x = self.fc1(x)
                x = self.relu(x)
                x = self.dropout(x)
                x = self.fc2(x)
                return x

        return PrunedMLP(self.model, pruned_layer_name, new_layer, new_next_layer).to(self.device)


class SimpleMLP(nn.Module):
    """Simple MLP for classification."""
    def __init__(self, input_size=784, hidden_size=256, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def train_model(model, train_loader, test_loader, device, epochs=10):
    """Train the model and return final accuracy."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    # Evaluate
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = 100 * correct / total
    return accuracy


def load_dataset(dataset_name):
    """Load specified dataset with appropriate transforms."""
    if dataset_name == 'MNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST('./data', train=False, transform=transform)
        input_size = 28 * 28
        num_classes = 10

    elif dataset_name == 'FashionMNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform)
        input_size = 28 * 28
        num_classes = 10

    elif dataset_name == 'CIFAR10':
        transform_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform_train)
        test_dataset = datasets.CIFAR10('./data', train=False, transform=transform_test)
        input_size = 32 * 32 * 3
        num_classes = 10

    elif dataset_name == 'KMNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train_dataset = datasets.KMNIST('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.KMNIST('./data', train=False, transform=transform)
        input_size = 28 * 28
        num_classes = 10

    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    return train_dataset, test_dataset, input_size, num_classes


def main(dataset_name='MNIST', prune_ratio=0.3):
    """Main execution: Train, prune, fine-tune, and compare."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    print(f"Dataset: {dataset_name}")

    # Load dataset
    train_dataset, test_dataset, input_size, num_classes = load_dataset(dataset_name)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

    # Train original model
    print("\n=== Training Original Model ===")
    original_model = SimpleMLP(input_size=input_size, hidden_size=256, num_classes=num_classes)
    original_accuracy = train_model(original_model, train_loader, test_loader, device, epochs=10)
    print(f"Original Model Accuracy: {original_accuracy:.2f}%")

    # Prune the model
    print("\n=== Pruning Model ===")
    pruner = SpectralPruner(original_model, device)
    pruned_model, importance_scores = pruner.prune_layer(
        layer_name='fc1',
        prune_ratio=prune_ratio,
        dataloader=train_loader,
        num_batches=20
    )

    # Evaluate pruned model before fine-tuning
    pruned_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = pruned_model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    before_finetune_accuracy = 100 * correct / total
    print(f"Pruned Model Accuracy (before fine-tuning): {before_finetune_accuracy:.2f}%")

    # Fine-tune pruned model
    print("\n=== Fine-tuning Pruned Model ===")
    finetuned_accuracy = train_model(pruned_model, train_loader, test_loader, device, epochs=5)
    print(f"Pruned Model Accuracy (after fine-tuning): {finetuned_accuracy:.2f}%")

    # Calculate model sizes
    original_params = sum(p.numel() for p in original_model.parameters())
    pruned_params = sum(p.numel() for p in pruned_model.parameters())
    compression_ratio = (1 - pruned_params / original_params) * 100

    print(f"\n=== Results Summary ===")
    print(f"Original Model: {original_params:,} parameters, {original_accuracy:.2f}% accuracy")
    print(f"Pruned Model: {pruned_params:,} parameters, {finetuned_accuracy:.2f}% accuracy")
    print(f"Compression: {compression_ratio:.1f}% reduction in parameters")
    print(f"Accuracy drop: {original_accuracy - finetuned_accuracy:.2f}%")


if __name__ == "__main__":
    # Test on multiple datasets
    datasets_to_test = ['MNIST', 'FashionMNIST', 'CIFAR10', 'KMNIST']
    prune_ratios = [0.2, 0.3, 0.4]  # Test different pruning levels

    print("=" * 80)
    print("SPECTRAL NETWORK PRUNING - MULTI-DATASET COMPARISON")
    print("=" * 80)

    for dataset_name in datasets_to_test:
        print(f"\n{'=' * 80}")
        print(f"DATASET: {dataset_name}")
        print(f"{'=' * 80}")

        for prune_ratio in prune_ratios:
            print(f"\n--- Prune Ratio: {prune_ratio*100:.0f}% ---")
            try:
                main(dataset_name=dataset_name, prune_ratio=prune_ratio)
            except Exception as e:
                print(f"Error with {dataset_name} at {prune_ratio}: {e}")
            print("\n")

SPECTRAL NETWORK PRUNING - MULTI-DATASET COMPARISON

DATASET: MNIST

--- Prune Ratio: 20% ---
Using device: cpu
Dataset: MNIST


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 478kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.43MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.42MB/s]



=== Training Original Model ===
Original Model Accuracy: 97.97%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 97.60%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 98.16%

=== Results Summary ===
Original Model: 203,530 parameters, 97.97% accuracy
Pruned Model: 162,190 parameters, 98.16% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: -0.19%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: MNIST

=== Training Original Model ===
Original Model Accuracy: 97.80%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 96.79%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 97.82%

=== Results Summary ===
Original Model: 203,530 parameters, 97.80% accuracy
Pruned Model: 142,315 parameters, 97.82% accuracy
Compression: 30.1% reduction in parameters
Accuracy drop: -0.02%

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 207kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.88MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.6MB/s]



=== Training Original Model ===
Original Model Accuracy: 86.76%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 86.08%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 87.06%

=== Results Summary ===
Original Model: 203,530 parameters, 86.76% accuracy
Pruned Model: 162,190 parameters, 87.06% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: -0.30%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: FashionMNIST

=== Training Original Model ===
Original Model Accuracy: 87.06%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 84.98%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 88.01%

=== Results Summary ===
Original Model: 203,530 parameters, 87.06% accuracy
Pruned Model: 142,315 parameters, 88.01% accuracy
Compression: 30.1% reduction in parameters
Accuracy drop:

100%|██████████| 170M/170M [00:03<00:00, 48.9MB/s]



=== Training Original Model ===
Original Model Accuracy: 46.62%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 44.82%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 48.96%

=== Results Summary ===
Original Model: 789,258 parameters, 46.62% accuracy
Pruned Model: 628,942 parameters, 48.96% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: -2.34%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: CIFAR10

=== Training Original Model ===
Original Model Accuracy: 48.28%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 42.43%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 49.74%

=== Results Summary ===
Original Model: 789,258 parameters, 48.28% accuracy
Pruned Model: 551,867 parameters, 49.74% accuracy
Compression: 30.1% reduction in parameters
Accuracy drop: -1.4

100%|██████████| 18.2M/18.2M [00:17<00:00, 1.05MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 329kB/s]
100%|██████████| 3.04M/3.04M [00:01<00:00, 2.30MB/s]
100%|██████████| 5.12k/5.12k [00:00<00:00, 7.97MB/s]



=== Training Original Model ===
Original Model Accuracy: 88.28%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 86.82%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 88.21%

=== Results Summary ===
Original Model: 203,530 parameters, 88.28% accuracy
Pruned Model: 162,190 parameters, 88.21% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: 0.07%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: KMNIST

=== Training Original Model ===
Original Model Accuracy: 87.72%

=== Pruning Model ===
Pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 85.84%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 87.56%

=== Results Summary ===
Original Model: 203,530 parameters, 87.72% accuracy
Pruned Model: 142,315 parameters, 87.56% accuracy
Compression: 30.1% reduction in parameters
Accuracy drop: 0.16%


In [ ]:
# -*- coding: utf-8 -*-
"""Network_pruning_on_Multiple_Datasets_Random.ipynb

This script performs random network pruning on various datasets
to serve as a baseline for comparison with structured pruning methods.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
from scipy.linalg import eigh
import copy

class RandomPruner:
    """
    Network pruning by randomly selecting neurons to remove.
    This serves as a baseline to compare against importance-based methods.
    """

    def __init__(self, model, device='cpu'):
        self.model = model
        self.device = device

    def prune_layer(self, layer_name, prune_ratio=0.2):
        """
        Prune a specific layer by randomly removing neurons.

        Args:
            layer_name (str): Name of the layer to prune (e.g., 'fc1').
            prune_ratio (float): Fraction of neurons to remove (0.0 to 1.0).

        Returns:
            torch.nn.Module: A new model with the specified layer pruned.
        """
        # Find the layer module to determine its size
        layer_to_prune = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                layer_to_prune = module
                break

        if layer_to_prune is None:
            raise ValueError(f"Layer '{layer_name}' not found in the model.")

        num_neurons = layer_to_prune.out_features
        num_to_keep = int(num_neurons * (1 - prune_ratio))

        # Randomly select indices of neurons to keep
        all_indices = np.arange(num_neurons)
        keep_indices = np.sort(np.random.choice(all_indices, num_to_keep, replace=False))

        print(f"Randomly pruning layer '{layer_name}': {num_neurons} -> {num_to_keep} neurons "
              f"({prune_ratio*100:.1f}% pruned)")

        # Create the pruned model using the same helper methods as the original script
        pruned_model = self._create_pruned_model(layer_name, keep_indices)

        return pruned_model

    def _create_pruned_model(self, layer_name, keep_indices):
        """
        Creates a new model with specified neurons removed from a layer.
        This method is kept structurally identical to your original version for a fair comparison.
        """
        # Get the layer to prune and the subsequent layer
        layer_to_prune = None
        next_layer = None
        found_target = False
        for name, module in self.model.named_modules():
            if name == layer_name:
                layer_to_prune = module
                found_target = True
            elif found_target and isinstance(module, nn.Linear):
                next_layer = module
                break

        if layer_to_prune is None:
            raise ValueError(f"Layer {layer_name} not found")

        # Create pruned weights and biases
        with torch.no_grad():
            # Prune the output dimension of the target layer
            pruned_weight = layer_to_prune.weight[keep_indices, :]
            pruned_bias = layer_to_prune.bias[keep_indices] if layer_to_prune.bias is not None else None

            # Create a new layer with the reduced output dimension
            new_layer = nn.Linear(
                layer_to_prune.in_features,
                len(keep_indices),
                bias=(pruned_bias is not None)
            ).to(self.device)
            new_layer.weight.copy_(pruned_weight)
            if pruned_bias is not None:
                new_layer.bias.copy_(pruned_bias)

            # Adjust the input dimension of the next layer
            new_next_layer = None
            if next_layer is not None:
                next_layer_weight = next_layer.weight[:, keep_indices]
                new_next_layer = nn.Linear(
                    len(keep_indices),
                    next_layer.out_features,
                    bias=(next_layer.bias is not None)
                ).to(self.device)
                new_next_layer.weight.copy_(next_layer_weight)
                if next_layer.bias is not None:
                    new_next_layer.bias.copy_(next_layer.bias)

        # Rebuild the model with the new, smaller layers
        pruned_model = self._rebuild_model(layer_name, new_layer, next_layer, new_next_layer)
        return pruned_model

    def _rebuild_model(self, pruned_layer_name, new_layer, old_next_layer, new_next_layer):
        """Rebuilds the model with the pruned layer, assuming a simple MLP structure."""

        class PrunedMLP(nn.Module):
            def __init__(self, original_model, pruned_name, new_l, new_next_l):
                super().__init__()
                self.flatten = nn.Flatten()

                # Copy layers from the original model, replacing the pruned ones
                if pruned_name == 'fc1':
                    self.fc1 = new_l
                    self.relu = nn.ReLU()
                    self.dropout = nn.Dropout(0.5)
                    self.fc2 = new_next_l if new_next_l else original_model.fc2
                else: # Fallback for pruning other layers, if needed
                    self.fc1 = original_model.fc1
                    self.relu = nn.ReLU()
                    self.dropout = nn.Dropout(0.5)
                    self.fc2 = original_model.fc2

            def forward(self, x):
                x = self.flatten(x)
                x = self.fc1(x)
                x = self.relu(x)
                x = self.dropout(x)
                x = self.fc2(x)
                return x

        return PrunedMLP(self.model, pruned_layer_name, new_layer, new_next_layer).to(self.device)


class SimpleMLP(nn.Module):
    """Simple MLP for classification."""
    def __init__(self, input_size=784, hidden_size=256, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def train_model(model, train_loader, test_loader, device, epochs=10):
    """Train the model and return final accuracy."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    # Evaluate
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = 100 * correct / total
    return accuracy


def load_dataset(dataset_name):
    """Load specified dataset with appropriate transforms."""
    if dataset_name == 'MNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST('./data', train=False, transform=transform)
        input_size = 28 * 28
        num_classes = 10

    elif dataset_name == 'FashionMNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.FashionMNIST('./data', train=False, transform=transform)
        input_size = 28 * 28
        num_classes = 10

    elif dataset_name == 'CIFAR10':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
        input_size = 32 * 32 * 3
        num_classes = 10

    elif dataset_name == 'KMNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train_dataset = datasets.KMNIST('./data', train=True, download=True, transform=transform)
        test_dataset = datasets.KMNIST('./data', train=False, transform=transform)
        input_size = 28 * 28
        num_classes = 10

    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    return train_dataset, test_dataset, input_size, num_classes


def main(dataset_name='MNIST', prune_ratio=0.3):
    """Main execution: Train, prune randomly, fine-tune, and compare."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    print(f"Dataset: {dataset_name}")

    # Load dataset
    train_dataset, test_dataset, input_size, num_classes = load_dataset(dataset_name)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

    # Train original model
    print("\n=== Training Original Model ===")
    original_model = SimpleMLP(input_size=input_size, hidden_size=256, num_classes=num_classes)
    original_accuracy = train_model(original_model, train_loader, test_loader, device, epochs=10)
    print(f"Original Model Accuracy: {original_accuracy:.2f}%")

    # Prune the model randomly
    print("\n=== Pruning Model Randomly ===")
    pruner = RandomPruner(original_model, device)
    pruned_model = pruner.prune_layer(
        layer_name='fc1',
        prune_ratio=prune_ratio,
    )

    # Evaluate pruned model before fine-tuning
    pruned_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = pruned_model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    before_finetune_accuracy = 100 * correct / total
    print(f"Pruned Model Accuracy (before fine-tuning): {before_finetune_accuracy:.2f}%")

    # Fine-tune pruned model
    print("\n=== Fine-tuning Pruned Model ===")
    finetuned_accuracy = train_model(pruned_model, train_loader, test_loader, device, epochs=5)
    print(f"Pruned Model Accuracy (after fine-tuning): {finetuned_accuracy:.2f}%")

    # Calculate model sizes
    original_params = sum(p.numel() for p in original_model.parameters())
    pruned_params = sum(p.numel() for p in pruned_model.parameters())
    compression_ratio = (1 - pruned_params / original_params) * 100

    print(f"\n=== Results Summary ===")
    print(f"Original Model: {original_params:,} parameters, {original_accuracy:.2f}% accuracy")
    print(f"Pruned Model: {pruned_params:,} parameters, {finetuned_accuracy:.2f}% accuracy")
    print(f"Compression: {compression_ratio:.1f}% reduction in parameters")
    print(f"Accuracy drop: {original_accuracy - finetuned_accuracy:.2f}%")


if __name__ == "__main__":
    datasets_to_test = ['MNIST', 'FashionMNIST', 'CIFAR10', 'KMNIST']
    prune_ratios = [0.2, 0.3, 0.4]

    print("=" * 80)
    print("RANDOM NETWORK PRUNING - MULTI-DATASET COMPARISON")
    print("=" * 80)

    for dataset_name in datasets_to_test:
        print(f"\n{'=' * 80}")
        print(f"DATASET: {dataset_name}")
        print(f"{'=' * 80}")

        for prune_ratio in prune_ratios:
            print(f"\n--- Prune Ratio: {prune_ratio*100:.0f}% ---")
            try:
                main(dataset_name=dataset_name, prune_ratio=prune_ratio)
            except Exception as e:
                print(f"An error occurred with {dataset_name} at {prune_ratio*100}% pruning: {e}")
            print("\n")

RANDOM NETWORK PRUNING - MULTI-DATASET COMPARISON

DATASET: MNIST

--- Prune Ratio: 20% ---
Using device: cpu
Dataset: MNIST


100%|██████████| 9.91M/9.91M [00:00<00:00, 65.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.73MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.03MB/s]



=== Training Original Model ===
Original Model Accuracy: 98.06%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 97.80%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 98.02%

=== Results Summary ===
Original Model: 203,530 parameters, 98.06% accuracy
Pruned Model: 162,190 parameters, 98.02% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: 0.04%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: MNIST

=== Training Original Model ===
Original Model Accuracy: 98.11%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 97.22%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 97.81%

=== Results Summary ===
Original Model: 203,530 parameters, 98.11% accuracy
Pruned Model: 142,315 parameters, 97.81% accuracy
Compression: 30.1% reduction 

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 304kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.60MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.9MB/s]



=== Training Original Model ===
Original Model Accuracy: 87.15%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 86.43%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 87.58%

=== Results Summary ===
Original Model: 203,530 parameters, 87.15% accuracy
Pruned Model: 162,190 parameters, 87.58% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: -0.43%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: FashionMNIST

=== Training Original Model ===
Original Model Accuracy: 87.38%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 86.80%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 87.40%

=== Results Summary ===
Original Model: 203,530 parameters, 87.38% accuracy
Pruned Model: 142,315 parameters, 87.40% accuracy
Compression: 30.1% re

100%|██████████| 170M/170M [00:03<00:00, 45.3MB/s]



=== Training Original Model ===
Original Model Accuracy: 47.61%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 46.17%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 47.97%

=== Results Summary ===
Original Model: 789,258 parameters, 47.61% accuracy
Pruned Model: 628,942 parameters, 47.97% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: -0.36%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: CIFAR10

=== Training Original Model ===
Original Model Accuracy: 46.09%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 43.59%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 49.00%

=== Results Summary ===
Original Model: 789,258 parameters, 46.09% accuracy
Pruned Model: 551,867 parameters, 49.00% accuracy
Compression: 30.1% reducti

100%|██████████| 18.2M/18.2M [00:14<00:00, 1.25MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 185kB/s]
100%|██████████| 3.04M/3.04M [00:01<00:00, 1.61MB/s]
100%|██████████| 5.12k/5.12k [00:00<00:00, 14.9MB/s]



=== Training Original Model ===
Original Model Accuracy: 88.11%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 204 neurons (20.0% pruned)
Pruned Model Accuracy (before fine-tuning): 87.18%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 87.47%

=== Results Summary ===
Original Model: 203,530 parameters, 88.11% accuracy
Pruned Model: 162,190 parameters, 87.47% accuracy
Compression: 20.3% reduction in parameters
Accuracy drop: 0.64%



--- Prune Ratio: 30% ---
Using device: cpu
Dataset: KMNIST

=== Training Original Model ===
Original Model Accuracy: 88.30%

=== Pruning Model Randomly ===
Randomly pruning layer 'fc1': 256 -> 179 neurons (30.0% pruned)
Pruned Model Accuracy (before fine-tuning): 84.97%

=== Fine-tuning Pruned Model ===
Pruned Model Accuracy (after fine-tuning): 87.01%

=== Results Summary ===
Original Model: 203,530 parameters, 88.30% accuracy
Pruned Model: 142,315 parameters, 87.01% accuracy
Compression: 30.1% reduction